# Elias Intelligence Research — Full Analysis Engine

**One-time setup:** Paste your GitHub Personal Access Token in the CONFIG cell.

**Then:** Runtime → Run all. Single `index.html` with all data embedded pushes directly to GitHub Pages.

View live at: https://ramezelias1.github.io/elias-intelligence-research/

In [ ]:
GITHUB_TOKEN  = 'PASTE_YOUR_TOKEN_HERE'
GITHUB_REPO   = 'ramezelias1/elias-intelligence-research'
GITHUB_BRANCH = 'main'
TICKERS = {'BHP':'BHP.AX','PLS':'PLS.AX','REA':'REA.AX','ZIP':'ZIP.AX','LTR':'LTR.AX','PLTR':'PLTR'}
ROLES   = {'BHP':'ASX Large-Cap','PLS':'ASX Mid-Cap','REA':'ASX Mid-Cap','ZIP':'ASX Small-Cap','LTR':'ASX Event Stock','PLTR':'US Comparison'}
START, END    = '2024-01-01', '2025-12-31'
MOVE_THRESH, MOVE_WINDOW, PRE_WINDOW, VOL_BASE = 0.15, 30, 15, 20
WEEKLY_SWING_MIN = 3
CS_THRESHOLD  = 55
print('Config ready.')

In [ ]:
import subprocess
subprocess.run(['pip','install','yfinance','pandas','numpy','scipy','-q'], capture_output=True)
import yfinance as yf
import pandas as pd
import numpy as np
import json, base64, requests
from datetime import datetime, timedelta
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
print('Libraries ready.')

In [ ]:
daily_data, weekly_data = {}, {}
for label, ticker in TICKERS.items():
    print(f'Fetching {label}...', end=' ', flush=True)
    try:
        df = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=True)
        if df.empty: print('SKIP'); continue
        if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
        df = df[['Open','High','Low','Close','Volume']].dropna()
        df['vol_avg20'] = df['Volume'].rolling(20).mean()
        df['vol_ratio'] = df['Volume'] / df['vol_avg20']
        daily_data[label] = df
        wdf = yf.download(ticker, start='2022-01-01', end=END, progress=False, auto_adjust=True, interval='1wk')
        if isinstance(wdf.columns, pd.MultiIndex): wdf.columns = wdf.columns.get_level_values(0)
        weekly_data[label] = wdf[['Open','High','Low','Close','Volume']].dropna()
        print(f'OK daily={len(df)} weekly={len(weekly_data[label])}')
    except Exception as e:
        print(f'ERROR {e}')
print(f'Fetched {len(daily_data)}/6 stocks.')

In [ ]:
def find_swing_points(wdf, lookback=3):
    highs, lows, dates = wdf['High'].values, wdf['Low'].values, wdf.index
    sh, sl = [], []
    for i in range(lookback, len(highs)-lookback):
        wh = highs[i-lookback:i+lookback+1]
        if highs[i] == max(wh) and list(wh).count(highs[i]) == 1: sh.append((dates[i], highs[i]))
        wl = lows[i-lookback:i+lookback+1]
        if lows[i] == min(wl) and list(wl).count(lows[i]) == 1: sl.append((dates[i], lows[i]))
    return sh, sl

def determine_trend_at_date(wdf, target_date, min_swings=WEEKLY_SWING_MIN):
    past = wdf[wdf.index <= pd.Timestamp(target_date)].tail(52)
    if len(past) < 10: return 'neutral', 0, 0, 0, 0
    sh, sl = find_swing_points(past)
    if len(sh) < 2 or len(sl) < 2: return 'neutral', 0, 0, 0, 0
    shv = [v for _, v in sh[-8:]]
    slv = [v for _, v in sl[-8:]]
    hh = sum(1 for i in range(1,len(shv)) if shv[i] > shv[i-1])
    hl = sum(1 for i in range(1,len(slv)) if slv[i] > slv[i-1])
    lh = sum(1 for i in range(1,len(shv)) if shv[i] < shv[i-1])
    ll = sum(1 for i in range(1,len(slv)) if slv[i] < slv[i-1])
    if hh >= min_swings and hl >= min_swings: return 'uptrend', hh, hl, lh, ll
    elif lh >= min_swings and ll >= min_swings: return 'downtrend', hh, hl, lh, ll
    return 'neutral', hh, hl, lh, ll

trend_states = {}
for label, wdf in weekly_data.items():
    state, hh, hl, lh, ll = determine_trend_at_date(wdf, END)
    trend_states[label] = {'state':state,'hh_count':hh,'hl_count':hl,'lh_count':lh,'ll_count':ll,
        'description':f'Last 8 weekly swings: {hh} HH, {hl} HL, {lh} LH, {ll} LL'}
    print(f'{label}: {state.upper():12} HH={hh} HL={hl} LH={lh} LL={ll}')
print('Trend detection complete.')

In [ ]:
def detect_moves(df):
    moves, used = [], set()
    closes, dates = df['Close'], df.index.tolist()
    for i in range(len(dates)-3):
        if i in used: continue
        sp = float(closes.iloc[i])
        cutoff = dates[i] + timedelta(days=MOVE_WINDOW)
        for j in range(i+1, len(dates)):
            if dates[j] > cutoff: break
            ep = float(closes.iloc[j])
            pct = (ep - sp) / sp
            if abs(pct) >= MOVE_THRESH:
                moves.append({'start_idx':i,'end_idx':j,'start_date':str(dates[i].date()),
                    'end_date':str(dates[j].date()),'start_price':round(sp,4),'end_price':round(ep,4),
                    'pct_move':round(pct*100,2),'direction':'LONG' if pct>0 else 'SHORT','trading_days':j-i})
                used.add(i); break
    if not moves: return []
    filtered, last = [], -10
    for m in sorted(moves, key=lambda x: x['start_idx']):
        if m['start_idx']-last > 5: filtered.append(m); last=m['start_idx']
        elif abs(m['pct_move']) > abs(filtered[-1]['pct_move']): filtered[-1]=m
    return filtered

def describe_candle(row):
    body = abs(float(row['Close'])-float(row['Open']))
    total = float(row['High'])-float(row['Low'])
    if total == 0: return 'no range'
    bp = body/total
    d = 'up' if float(row['Close'])>=float(row['Open']) else 'down'
    uw = (float(row['High'])-max(float(row['Open']),float(row['Close'])))/total
    lw = (min(float(row['Open']),float(row['Close']))-float(row['Low']))/total
    parts = []
    if bp>0.7: parts.append(f'strong {d} body {bp:.0%}')
    elif bp>0.4: parts.append(f'moderate {d} body {bp:.0%}')
    else: parts.append(f'indecision {bp:.0%}')
    if uw>0.35: parts.append(f'upper wick {uw:.0%}')
    if lw>0.35: parts.append(f'lower wick {lw:.0%}')
    parts.append(f'range {(total/max(float(row["Open"]),0.001))*100:.1f}pct')
    return ' | '.join(parts)

def describe_vol(vr):
    if vr is None or (isinstance(vr,float) and np.isnan(vr)): return 'no baseline'
    if vr>=4: return f'{vr:.1f}x extreme'
    if vr>=2.5: return f'{vr:.1f}x major surge'
    if vr>=1.5: return f'{vr:.1f}x elevated'
    if vr>=0.8: return f'{vr:.1f}x normal'
    return f'{vr:.1f}x drying up'

def compression_score(window_df):
    if window_df.empty or len(window_df) < 5: return 0
    candles = [describe_candle(row) for _, row in window_df.iterrows()]
    n = len(candles)
    c1 = sum(1 for c in candles if 'indecision' in c) / n * 35
    uw = sum(1 for c in candles if 'upper wick' in c) / n
    lw = sum(1 for c in candles if 'lower wick' in c) / n
    c2 = min(uw, lw) * 2 * 30
    vols = window_df['vol_ratio'].dropna().tolist()
    c3 = (sum(1 for v in vols if 0.7<=v<=1.3)/len(vols)*20) if vols else 10
    ranges = [(float(r['High'])-float(r['Low']))/max(float(r['Close']),0.001) for _,r in window_df.iterrows()]
    if len(ranges)>=6:
        er, lr2 = np.mean(ranges[:5]), np.mean(ranges[-5:])
        c4 = min(max(0,(er-lr2)/(er+1e-9))*2,1)*15
    else: c4=7
    return round(min(c1+c2+c3+c4,100),1)

def summarise(pre_df):
    if pre_df.empty: return 'No pre-move data.'
    vols = pre_df['vol_ratio'].dropna().tolist()
    if not vols: return 'No volume baseline.'
    n = len(pre_df)
    early = np.mean(vols[:5]) if len(vols)>=5 else np.mean(vols)
    late  = np.mean(vols[-5:]) if len(vols)>=5 else np.mean(vols)
    trend = 'RISING' if late>early*1.2 else 'FALLING' if late<early*0.8 else 'STABLE'
    drift = ((float(pre_df['Close'].iloc[-1])-float(pre_df['Close'].iloc[0]))/float(pre_df['Close'].iloc[0]))*100
    hv = int((pre_df['vol_ratio']>=1.5).sum())
    descs = [describe_candle(r) for _,r in pre_df.iterrows()]
    parts = [f'Drift: {drift:+.1f}% over {n} days.',
             f'Volume: {trend} (early {early:.1f}x to late {late:.1f}x).',
             f'Elevated vol days: {hv}/{n}.']
    su=sum(1 for d in descs if 'strong up' in d)
    sd=sum(1 for d in descs if 'strong down' in d)
    ind=sum(1 for d in descs if 'indecision' in d)
    ur=sum(1 for d in descs if 'upper wick' in d)
    lr=sum(1 for d in descs if 'lower wick' in d)
    if su: parts.append(f'Strong up candles: {su}.')
    if sd: parts.append(f'Strong down candles: {sd}.')
    if ind: parts.append(f'Indecision candles: {ind}.')
    if ur: parts.append(f'Upper wick rejections: {ur}.')
    if lr: parts.append(f'Lower wick rejections: {lr}.')
    return ' '.join(parts)

print('Functions ready.')

In [ ]:
stocks_results = {}
for label, df in daily_data.items():
    wdf = weekly_data.get(label)
    moves = detect_moves(df)
    print(f'{label}: {len(moves)} moves')
    analysed = []
    for move in moves:
        pre_df   = df.iloc[max(0,move['start_idx']-PRE_WINDOW):move['start_idx']].copy()
        trig_row = df.iloc[move['start_idx']]
        cs = compression_score(pre_df)
        t_state, hh, hl, lh, ll = determine_trend_at_date(wdf, move['start_date']) if wdf is not None else ('neutral',0,0,0,0)
        pre_days = []
        for di,(date,row) in enumerate(pre_df.iterrows()):
            vr = float(row['vol_ratio']) if not pd.isna(row['vol_ratio']) else None
            pre_days.append({'day_offset':di-len(pre_df)+1,'date':str(date.date()),
                'open':round(float(row['Open']),4),'high':round(float(row['High']),4),
                'low':round(float(row['Low']),4),'close':round(float(row['Close']),4),
                'volume':int(row['Volume']),'vol_ratio':round(vr,2) if vr else None,
                'candle_desc':describe_candle(row),'volume_desc':describe_vol(vr)})
        tvr = float(trig_row['vol_ratio']) if not pd.isna(trig_row['vol_ratio']) else None
        trig = {'date':str(df.index[move['start_idx']].date()),
            'open':round(float(trig_row['Open']),4),'high':round(float(trig_row['High']),4),
            'low':round(float(trig_row['Low']),4),'close':round(float(trig_row['Close']),4),
            'volume':int(trig_row['Volume']),'vol_ratio':round(tvr,2) if tvr else None,
            'candle_desc':describe_candle(trig_row),'volume_desc':describe_vol(tvr)}
        analysed.append({'move':move,'trigger_day':trig,'pre_days':pre_days,
            'summary':summarise(pre_df),'compression_score':cs,
            'trend_state':t_state,'trend_hh':hh,'trend_hl':hl,'trend_lh':lh,'trend_ll':ll})
    stocks_results[label] = {'label':label,'ticker':TICKERS[label],'role':ROLES[label],
        'total_days':len(df),'date_range':f"{df.index[0].date()} to {df.index[-1].date()}",
        'moves':analysed}
print('Move analysis complete.')

In [ ]:
backtest_signals = []
for label, df in daily_data.items():
    wdf = weekly_data.get(label)
    dates, closes = df.index.tolist(), df['Close']
    print(f'Backtesting {label}...', end=' ', flush=True)
    n_sig = 0
    for i in range(PRE_WINDOW, len(dates)-5):
        cs = compression_score(df.iloc[i-PRE_WINDOW:i])
        if cs < CS_THRESHOLD: continue
        signal_date = str(dates[i].date())
        t_state = determine_trend_at_date(wdf, signal_date)[0] if wdf is not None else 'neutral'
        sp = float(closes.iloc[i])
        cutoff = dates[i] + timedelta(days=MOVE_WINDOW)
        outcome, best_pct, direction = 'no_move', 0.0, None
        for j in range(i+1, min(i+35,len(dates))):
            if dates[j] > cutoff: break
            pct = (float(closes.iloc[j]) - sp) / sp
            if abs(pct) > abs(best_pct): best_pct = pct
            if abs(pct) >= MOVE_THRESH:
                direction = 'LONG' if pct>0 else 'SHORT'
                outcome = direction; break
        aligned = (t_state=='uptrend' and direction=='LONG') or (t_state=='downtrend' and direction=='SHORT')
        backtest_signals.append({'label':label,'signal_date':signal_date,'cs':cs,
            'trend_state':t_state,'outcome':outcome,'direction':direction,
            'best_pct':round(best_pct*100,2),'aligned':aligned})
        n_sig += 1
    print(f'{n_sig} signals')

total = len(backtest_signals)
hits  = [s for s in backtest_signals if s['outcome'] != 'no_move']
al_s  = [s for s in backtest_signals if s['trend_state'] != 'neutral']
al_h  = [s for s in al_s if s['aligned']]
sr_any = round(len(hits)/total*100,1) if total else 0
sr_al  = round(len(al_h)/len(al_s)*100,1) if al_s else 0
avg_m  = round(np.mean([abs(s['best_pct']) for s in hits]),2) if hits else 0
fr     = round((1-len(hits)/total)*100,1) if total else 0
by_stock = {}
for label in TICKERS:
    ss=[s for s in backtest_signals if s['label']==label]
    sh=[s for s in ss if s['outcome']!='no_move']
    sa=[s for s in ss if s['trend_state']!='neutral']
    sah=[s for s in sa if s['aligned']]
    by_stock[label]={'total_signals':len(ss),
        'success_rate':round(len(sh)/len(ss)*100,1) if ss else 0,
        'trend_aligned_rate':round(len(sah)/len(sa)*100,1) if sa else 0,
        'avg_move':round(np.mean([abs(s['best_pct']) for s in sh]),2) if sh else 0}
print(f'Total={total} Success={sr_any}% TrendAligned={sr_al}% AvgMove={avg_m}% FalseRate={fr}%')

In [ ]:
# ── Build self-contained HTML with all data embedded and push to GitHub ──
all_moves = [m for s in stocks_results.values() for m in s['moves']]
avg_cs = round(np.mean([m['compression_score'] for m in all_moves]),1) if all_moves else 0
longs  = sum(1 for m in all_moves if m['move']['direction']=='LONG')
shorts = sum(1 for m in all_moves if m['move']['direction']=='SHORT')
gen    = datetime.now().strftime('%d %b %Y %H:%M')

results = {
    'generated_at': gen,
    'stocks': stocks_results,
    'trends': trend_states,
    'backtest': {
        'total_signals': total,
        'success_rate_any': sr_any,
        'success_rate_trend_aligned': sr_al,
        'avg_move_after': avg_m,
        'false_signal_rate': fr,
        'cs_threshold': CS_THRESHOLD,
        'by_stock': by_stock,
    },
    'compression_stats': {'avg_score': avg_cs, 'total_longs': longs, 'total_shorts': shorts}
}

rj = json.dumps(results, default=str)

html = '''<!DOCTYPE html>
<html lang="en">
<head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Elias Intelligence Research</title>
<link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=JetBrains+Mono:wght@300;400;500&display=swap" rel="stylesheet">
<style>
:root{--bg:#07090c;--s1:#0f1318;--s2:#161b22;--b:#1e2530;--g:#00e5b0;--r:#ff4d6d;--y:#ffd166;--p:#7c6fff;--t:#e8eaf0;--t2:#8892a0;--t3:#3d4a5c;}
*{margin:0;padding:0;box-sizing:border-box;}
body{background:var(--bg);color:var(--t);font-family:JetBrains Mono,monospace;font-size:12px;min-height:100vh;}
.hdr{background:var(--s1);border-bottom:1px solid var(--b);padding:18px 28px;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;position:sticky;top:0;z-index:100;}
.hdr h1{font-family:Syne,sans-serif;font-size:18px;font-weight:800;color:var(--g);}
.hdr p{color:var(--t2);font-size:10px;margin-top:2px;text-transform:uppercase;letter-spacing:1px;}
.badges{display:flex;gap:5px;flex-wrap:wrap;}
.badge{padding:3px 8px;border-radius:3px;font-size:9px;text-transform:uppercase;letter-spacing:.5px;}
.bg{background:rgba(0,229,176,.1);color:var(--g);border:1px solid rgba(0,229,176,.25);}
.by{background:rgba(255,209,102,.1);color:var(--y);border:1px solid rgba(255,209,102,.25);}
.br{background:rgba(255,77,109,.1);color:var(--r);border:1px solid rgba(255,77,109,.25);}
.bp{background:rgba(124,111,255,.1);color:var(--p);border:1px solid rgba(124,111,255,.25);}
.main{padding:22px 28px;}
.tabs{display:flex;gap:2px;margin-bottom:20px;border-bottom:1px solid var(--b);}
.tab{padding:8px 16px;cursor:pointer;font-size:11px;color:var(--t3);border-bottom:2px solid transparent;transition:all .15s;margin-bottom:-1px;}
.tab.act{color:var(--g);border-bottom-color:var(--g);}
.tc{display:none;}.tc.act{display:block;}
.info{background:rgba(0,229,176,.05);border:1px solid rgba(0,229,176,.2);border-radius:5px;padding:11px 15px;margin-bottom:18px;font-size:11px;color:var(--t2);line-height:1.8;}
.info strong{color:var(--g);}
.stats{display:grid;grid-template-columns:repeat(auto-fill,minmax(140px,1fr));gap:9px;margin-bottom:20px;}
.stat{background:var(--s1);border:1px solid var(--b);border-radius:5px;padding:11px 13px;}
.sl{font-size:9px;text-transform:uppercase;letter-spacing:1px;color:var(--t3);margin-bottom:3px;}
.sv{font-family:Syne,sans-serif;font-size:19px;font-weight:700;}
.ss{font-size:10px;color:var(--t2);margin-top:2px;}
.filters{display:flex;gap:7px;flex-wrap:wrap;margin-bottom:16px;align-items:center;}
.fl{color:var(--t3);font-size:10px;text-transform:uppercase;letter-spacing:1px;}
.fb{padding:4px 10px;border:1px solid var(--b);background:transparent;color:var(--t2);border-radius:3px;cursor:pointer;font-family:JetBrains Mono,monospace;font-size:11px;transition:all .15s;}
.fb:hover,.fb.act{border-color:var(--g);color:var(--g);background:rgba(0,229,176,.08);}
.sec{font-family:Syne,sans-serif;font-size:13px;font-weight:700;color:var(--t);margin-bottom:13px;padding-bottom:7px;border-bottom:1px solid var(--b);}
.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(320px,1fr));gap:13px;}
.mc{background:var(--s1);border:1px solid var(--b);border-radius:6px;overflow:hidden;}
.mch{padding:11px 15px;display:flex;justify-content:space-between;align-items:center;cursor:pointer;border-bottom:1px solid var(--b);transition:background .15s;}
.mch:hover{background:rgba(0,229,176,.03);}
.mt{font-family:Syne,sans-serif;font-size:14px;font-weight:800;}
.md{font-size:9px;font-weight:600;padding:2px 6px;border-radius:3px;}
.dl{color:var(--g);background:rgba(0,229,176,.12);}.ds{color:var(--r);background:rgba(255,77,109,.12);}
.tu{color:var(--g);background:rgba(0,229,176,.08);border:1px solid rgba(0,229,176,.2);padding:2px 6px;border-radius:3px;font-size:9px;}
.td{color:var(--r);background:rgba(255,77,109,.08);border:1px solid rgba(255,77,109,.2);padding:2px 6px;border-radius:3px;font-size:9px;}
.tn{color:var(--y);background:rgba(255,209,102,.08);border:1px solid rgba(255,209,102,.2);padding:2px 6px;border-radius:3px;font-size:9px;}
.mp{font-size:13px;font-weight:600;}
.mdt{padding:6px 15px;font-size:10px;color:var(--t2);background:rgba(0,0,0,.2);border-bottom:1px solid var(--b);}
.mb{display:none;}.mb.open{display:block;}
.ms{padding:11px 15px;font-size:10px;color:var(--t2);line-height:1.8;border-bottom:1px solid var(--b);}
.csb{margin:8px 15px;background:var(--s2);border-radius:4px;overflow:hidden;height:5px;}
.csf{height:100%;border-radius:4px;}
.csl{display:flex;justify-content:space-between;padding:2px 15px 8px;font-size:9px;color:var(--t3);}
.tb{margin:9px 15px 11px;background:rgba(255,209,102,.05);border:1px solid rgba(255,209,102,.2);border-radius:4px;padding:9px 11px;}
.tt{font-size:9px;text-transform:uppercase;letter-spacing:1px;color:var(--y);margin-bottom:5px;}
.tr{display:flex;gap:13px;flex-wrap:wrap;margin-bottom:5px;}
.ti label{display:block;font-size:9px;color:var(--t3);text-transform:uppercase;}
.ti span{font-size:11px;color:var(--t);}
.tca{font-size:10px;color:var(--y);}
.tw{overflow-x:auto;padding:0 15px 13px;}
table{width:100%;border-collapse:collapse;font-size:10px;}
th{padding:4px 7px;text-align:left;color:var(--t3);font-size:9px;text-transform:uppercase;border-bottom:1px solid var(--b);font-weight:400;}
td{padding:3px 7px;border-bottom:1px solid rgba(30,37,48,.5);vertical-align:top;}
tr:last-child td{border-bottom:none;}
.ve{color:var(--r);}.vh{color:var(--y);}.vn{color:var(--t2);}.vl{color:var(--t3);}
.cu{color:var(--g);}.cd{color:var(--r);}.cn{color:var(--y);}
.tgrid{display:grid;grid-template-columns:repeat(auto-fill,minmax(260px,1fr));gap:12px;margin-bottom:20px;}
.tcard{background:var(--s1);border:1px solid var(--b);border-radius:6px;padding:14px 16px;}
.btsum{background:var(--s1);border:1px solid var(--b);border-radius:6px;padding:16px 18px;margin-bottom:18px;}
.bttit{font-family:Syne,sans-serif;font-size:14px;font-weight:700;margin-bottom:12px;color:var(--t);}
.btgrid{display:grid;grid-template-columns:repeat(auto-fill,minmax(130px,1fr));gap:8px;}
.btst{padding:9px 11px;background:var(--s2);border-radius:4px;border:1px solid var(--b);}
.btl{font-size:9px;text-transform:uppercase;letter-spacing:1px;color:var(--t3);margin-bottom:3px;}
.btv{font-family:Syne,sans-serif;font-size:16px;font-weight:700;}
.none{padding:18px;text-align:center;color:var(--t3);grid-column:1/-1;}
::-webkit-scrollbar{width:4px;height:4px;}::-webkit-scrollbar-track{background:var(--bg);}::-webkit-scrollbar-thumb{background:var(--b);border-radius:3px;}
</style></head>
<body>
<div class="hdr">
  <div><h1>ELIAS INTELLIGENCE RESEARCH</h1><p id="gt">Loading...</p></div>
  <div class="badges">
    <span class="badge bg" id="b1">—</span>
    <span class="badge by">2024-2025</span>
    <span class="badge br" id="b2">—</span>
    <span class="badge bp">Wealth Within Trend</span>
  </div>
</div>
<div class="main">
<div class="tabs">
  <div class="tab act" onclick="showTab(this,\'moves\')">Structural Moves</div>
  <div class="tab" onclick="showTab(this,\'trend\')">Trend Analysis</div>
  <div class="tab" onclick="showTab(this,\'backtest\')">Backtest Results</div>
</div>
<div class="tc act" id="tab-moves"></div>
<div class="tc" id="tab-trend"></div>
<div class="tc" id="tab-backtest"></div>
</div>
<script>
const D=__DATA__;
const allM=Object.values(D.stocks||{}).flatMap(s=>(s.moves||[]).map(m=>({...m.move,label:s.label,role:s.role,trend_state:m.trend_state||\'neutral\',compression_score:m.compression_score||0,summary:m.summary||\'\',pre_days:m.pre_days||[],trigger_day:m.trigger_day||{}})));
const longs=allM.filter(m=>m.direction===\'LONG\').length;
const shorts=allM.filter(m=>m.direction===\'SHORT\').length;
const avg=allM.length?(allM.reduce((s,m)=>s+Math.abs(m.pct_move),0)/allM.length).toFixed(1):0;
document.getElementById(\'gt\').textContent=\'Generated: \'+D.generated_at+\' — Raw OHLCV — Wealth Within Trend\';
document.getElementById(\'b1\').textContent=Object.keys(D.stocks||{}).length+\' Stocks\';
document.getElementById(\'b2\').textContent=allM.length+\' Moves\';
let aDir=\'all\',aSt=\'all\',aTr=\'all\';
function vc(r){if(!r)return \'vn\';if(r>=3)return \'ve\';if(r>=1.5)return \'vh\';if(r<0.7)return \'vl\';return \'vn\';}
function cc(d){if((d||\'\').includes(\'strong up\')||(d||\'\').includes(\'moderate up\'))return \'cu\';if((d||\'\').includes(\'strong down\')||(d||\'\').includes(\'moderate down\'))return \'cd\';return \'cn\';}
function tclass(t){return t===\'uptrend\'?\'tu\':t===\'downtrend\'?\'td\':\'tn\';}
function tlabel(t){return t===\'uptrend\'?\'\u2191 Uptrend\':t===\'downtrend\'?\'\u2193 Downtrend\':\'\u2192 Neutral\';}
function renderMoves(){
  const f=allM.filter(m=>(aDir===\'all\'||m.direction===aDir)&&(aSt===\'all\'||m.label===aSt)&&(aTr===\'all\'||m.trend_state===aTr));
  const el=document.getElementById(\'tab-moves\');
  const bt=D.backtest||{};
  el.innerHTML=`<div class="info"><strong>Methodology:</strong> Moves >=15% within 30 days. 15-day pre-move window. Volume baseline=20-day rolling avg. <strong>Zero TA indicators.</strong> Compression Score = structural tension (0-100). Trend = Wealth Within swing high/low definition on weekly data.</div>
  <div class="stats">
    <div class="stat"><div class="sl">Total Moves</div><div class="sv" style="color:var(--g)">${allM.length}</div><div class="ss">>=15% in 30 days</div></div>
    <div class="stat"><div class="sl">Long</div><div class="sv" style="color:var(--g)">${longs}</div><div class="ss">${allM.length?(longs/allM.length*100).toFixed(0):0}% of total</div></div>
    <div class="stat"><div class="sl">Short</div><div class="sv" style="color:var(--r)">${shorts}</div><div class="ss">${allM.length?(shorts/allM.length*100).toFixed(0):0}% of total</div></div>
    <div class="stat"><div class="sl">Avg Move</div><div class="sv" style="color:var(--y)">${avg}%</div><div class="ss">across all moves</div></div>
    <div class="stat"><div class="sl">Avg Compression</div><div class="sv" style="color:var(--p)">${(D.compression_stats||{}).avg_score||0}</div><div class="ss">0-100 scale</div></div>
  </div>
  <div class="filters">
    <span class="fl">Direction:</span>
    <button class="fb act" onclick="fil(\'dir\',\'all\',this)">All</button>
    <button class="fb" onclick="fil(\'dir\',\'LONG\',this)">Long</button>
    <button class="fb" onclick="fil(\'dir\',\'SHORT\',this)">Short</button>
    <span class="fl" style="margin-left:10px">Trend:</span>
    <button class="fb act" onclick="fil(\'tr\',\'all\',this)">All</button>
    <button class="fb" onclick="fil(\'tr\',\'uptrend\',this)">Uptrend</button>
    <button class="fb" onclick="fil(\'tr\',\'downtrend\',this)">Downtrend</button>
    <button class="fb" onclick="fil(\'tr\',\'neutral\',this)">Neutral</button>
    <span class="fl" style="margin-left:10px">Stock:</span>
    <button class="fb act" onclick="fil(\'st\',\'all\',this)">All</button>
    ${Object.keys(D.stocks||{}).map(l=>`<button class="fb" onclick="fil(\'st\',\'${l}\',this)">${l}</button>`).join(\'\')}
  </div>
  <div class="sec">Qualifying Moves — Click to expand</div>
  <div class="grid">${f.map((move,i)=>{
    const sd=D.stocks[move.label];
    const am=(sd.moves||[]).find(m=>m.move.start_date===move.start_date&&m.move.direction===move.direction);
    if(!am)return \'\' ;
    const col=move.direction===\'LONG\'?\'var(--g)\':\'var(--r)\';
    const dc=move.direction===\'LONG\'?\'dl\':\'ds\';
    const ps=(move.pct_move>0?\'+\':\'\')+move.pct_move+\'%\';
    const td=am.trigger_day||{};
    const cs=move.compression_score||0;
    const csc=cs>=70?\'var(--g)\':(cs>=50?\'var(--y)\':\'var(--r)\');
    const rows=(am.pre_days||[]).map(d=>`<tr><td style="color:var(--t3);font-size:9px">${d.day_offset}</td><td>${d.date}</td><td>${d.close}</td><td class="${vc(d.vol_ratio)}">${d.vol_ratio?d.vol_ratio+\'x\':\'--\'}</td><td class="${vc(d.vol_ratio)}" style="font-size:9px">${d.volume_desc||\'\'}</td><td class="${cc(d.candle_desc)}" style="font-size:9px">${d.candle_desc||\'\'}</td></tr>`).join(\'\');
    return `<div class="mc" data-dir="${move.direction}" data-st="${move.label}" data-tr="${move.trend_state}">
      <div class="mch" onclick="tog(${i})">
        <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap">
          <span class="mt" style="color:${col}">${move.label}</span>
          <span class="md ${dc}">${move.direction}</span>
          <span class="${tclass(move.trend_state)}">${tlabel(move.trend_state)}</span>
        </div>
        <div style="display:flex;align-items:center;gap:8px">
          <span class="mp" style="color:${col}">${ps}</span>
          <span id="cv-${i}" style="color:var(--t3)">+</span>
        </div>
      </div>
      <div class="mdt">${move.start_date} to ${move.end_date} &nbsp;·&nbsp; $${move.start_price} to $${move.end_price} &nbsp;·&nbsp; ${move.trading_days} days</div>
      <div class="mb" id="mb-${i}">
        <div class="csl"><span>Compression Score</span><span style="color:${csc}">${cs}/100</span></div>
        <div class="csb"><div class="csf" style="width:${cs}%;background:${csc}"></div></div>
        <div class="ms">${am.summary||\'\'}</div>
        <div class="tb">
          <div class="tt">Trigger Day — ${td.date||\'\'}</div>
          <div class="tr">
            <div class="ti"><label>Open</label><span>$${td.open||\'\'}</span></div>
            <div class="ti"><label>High</label><span>$${td.high||\'\'}</span></div>
            <div class="ti"><label>Low</label><span>$${td.low||\'\'}</span></div>
            <div class="ti"><label>Close</label><span>$${td.close||\'\'}</span></div>
            <div class="ti"><label>Vol Ratio</label><span class="${vc(td.vol_ratio)}">${td.vol_ratio?td.vol_ratio+\'x\':\'--\'}</span></div>
          </div>
          <div class="tca">${td.candle_desc||\'\'}</div>
        </div>
        <div class="tw"><table><thead><tr><th>Day</th><th>Date</th><th>Close</th><th>Vol Ratio</th><th>Vol Character</th><th>Candle Character</th></tr></thead><tbody>${rows}</tbody></table></div>
      </div>
    </div>`;
  }).join(\'\')}</div>`;
}
function renderTrend(){
  const el=document.getElementById(\'tab-trend\');
  const upM=allM.filter(m=>m.trend_state===\'uptrend\');
  const dnM=allM.filter(m=>m.trend_state===\'downtrend\');
  const ntM=allM.filter(m=>m.trend_state===\'neutral\');
  const upL=upM.filter(m=>m.direction===\'LONG\').length;
  const dnS=dnM.filter(m=>m.direction===\'SHORT\').length;
  const aln=allM.length?((upL+dnS)/allM.length*100).toFixed(1):0;
  el.innerHTML=`<div class="info"><strong>Wealth Within Trend Definition:</strong> Uptrend = 3+ higher swing highs AND 3+ higher swing lows on weekly chart. Downtrend = 3+ lower swing highs AND 3+ lower swing lows. Neutral = transitioning or insufficient data.</div>
  <div class="sec">Current Trend State per Stock</div>
  <div class="tgrid">${Object.entries(D.trends||{}).map(([label,t])=>{
    const sc=t.state===\'uptrend\'?\'tu\':t.state===\'downtrend\'?\'td\':\'tn\';
    const sl=t.state===\'uptrend\'?\'\u2191 UPTREND\':t.state===\'downtrend\'?\'\u2193 DOWNTREND\':\'\u2192 NEUTRAL\';
    return `<div class="tcard"><div style="font-family:Syne,sans-serif;font-size:16px;font-weight:800;color:var(--g);margin-bottom:8px">${label}</div>
      <span class="${sc}" style="display:inline-block;padding:4px 10px;border-radius:4px;font-size:11px;font-weight:600;margin-bottom:10px">${sl}</span>
      <div style="font-size:10px;color:var(--t2);line-height:1.9">
        <div>HH: <strong style="color:var(--t)">${t.hh_count||0}</strong> &nbsp; HL: <strong style="color:var(--t)">${t.hl_count||0}</strong> &nbsp; LH: <strong style="color:var(--t)">${t.lh_count||0}</strong> &nbsp; LL: <strong style="color:var(--t)">${t.ll_count||0}</strong></div>
        <div style="margin-top:4px">${t.description||\'\'}</div>
      </div>
    </div>`;
  }).join(\'\')}</div>
  <div class="sec">Trend-Direction Alignment</div>
  <div class="btsum"><div class="btgrid">
    <div class="btst"><div class="btl">Uptrend Moves</div><div class="btv" style="color:var(--g)">${upM.length}</div></div>
    <div class="btst"><div class="btl">Long in Uptrend</div><div class="btv" style="color:var(--g)">${upL}/${upM.length}</div></div>
    <div class="btst"><div class="btl">Downtrend Moves</div><div class="btv" style="color:var(--r)">${dnM.length}</div></div>
    <div class="btst"><div class="btl">Short in Downtrend</div><div class="btv" style="color:var(--r)">${dnS}/${dnM.length}</div></div>
    <div class="btst"><div class="btl">Neutral Moves</div><div class="btv" style="color:var(--y)">${ntM.length}</div></div>
    <div class="btst"><div class="btl">Alignment Rate</div><div class="btv" style="color:var(--p)">${aln}%</div></div>
  </div></div>`;
}
function renderBacktest(){
  const bt=D.backtest||{};
  const el=document.getElementById(\'tab-backtest\');
  el.innerHTML=`<div class="info"><strong>Backtest Methodology:</strong> Every day scanned for compression signal (score>=${bt.cs_threshold||55}). For each signal day, outcome checked over next 30 days. Success = 15%+ move occurred. Trend-aligned = move direction matched Wealth Within trend state.</div>
  <div class="btsum"><div class="bttit">Compression Signal Backtest — Full 2024-2025 Dataset</div><div class="btgrid">
    <div class="btst"><div class="btl">Total Signals</div><div class="btv" style="color:var(--g)">${bt.total_signals||0}</div></div>
    <div class="btst"><div class="btl">Success Rate (any)</div><div class="btv" style="color:var(--g)">${bt.success_rate_any||0}%</div></div>
    <div class="btst"><div class="btl">Trend Aligned Rate</div><div class="btv" style="color:var(--p)">${bt.success_rate_trend_aligned||0}%</div></div>
    <div class="btst"><div class="btl">Avg Move After</div><div class="btv" style="color:var(--y)">${bt.avg_move_after||0}%</div></div>
    <div class="btst"><div class="btl">False Signal Rate</div><div class="btv" style="color:var(--r)">${bt.false_signal_rate||0}%</div></div>
    <div class="btst"><div class="btl">CS Threshold</div><div class="btv" style="color:var(--t2)">${bt.cs_threshold||55}</div></div>
  </div></div>
  <div class="sec">Results by Stock</div>
  <div class="grid">${Object.entries(bt.by_stock||{}).map(([label,s])=>`
    <div class="mc"><div class="mch" style="cursor:default">
      <span class="mt" style="color:var(--g)">${label}</span>
      <span style="font-size:10px;color:var(--t2)">${s.total_signals||0} signals</span>
    </div>
    <div style="padding:10px 15px;font-size:10px;color:var(--t2);line-height:2">
      <div>Success rate: <strong style="color:var(--g)">${s.success_rate||0}%</strong></div>
      <div>Trend-aligned rate: <strong style="color:var(--p)">${s.trend_aligned_rate||0}%</strong></div>
      <div>Avg move after signal: <strong style="color:var(--y)">${s.avg_move||0}%</strong></div>
    </div></div>`).join(\'\')}</div>`;
}
function tog(i){
  const b=document.getElementById(\'mb-\'+i);
  const c=document.getElementById(\'cv-\'+i);
  b.classList.toggle(\'open\',!b.classList.contains(\'open\'));
  c.textContent=b.classList.contains(\'open\')?\'-\':\'+\';
}
function fil(type,val,btn){
  if(type===\'dir\'){aDir=val;document.querySelectorAll(\'.filters .fb\').forEach(b=>{if([\'All\',\'Long\',\'Short\'].includes(b.textContent))b.classList.remove(\'act\');});}
  else if(type===\'tr\'){aTr=val;document.querySelectorAll(\'.filters .fb\').forEach(b=>{if([\'All\',\'Uptrend\',\'Downtrend\',\'Neutral\'].includes(b.textContent))b.classList.remove(\'act\');});}
  else{aSt=val;document.querySelectorAll(\'.filters .fb\').forEach(b=>{if(b.textContent===\'All\'||Object.keys(D.stocks||{}).includes(b.textContent))b.classList.remove(\'act\');});}
  btn.classList.add(\'act\');renderMoves();
}
function showTab(btn,name){
  document.querySelectorAll(\'.tab\').forEach(t=>t.classList.remove(\'act\'));
  document.querySelectorAll(\'.tc\').forEach(t=>t.classList.remove(\'act\'));
  btn.classList.add(\'act\');
  document.getElementById(\'tab-\'+name).classList.add(\'act\');
}
renderMoves();renderTrend();renderBacktest();
</script></body></html>'''

html = html.replace('__DATA__', rj)
print(f'HTML size: {len(html):,} characters')

def push_to_github(filename, content, token, repo, branch='main'):
    url = f'https://api.github.com/repos/{repo}/contents/{filename}'
    hdrs = {'Authorization': f'token {token}', 'Accept': 'application/vnd.github.v3+json'}
    r = requests.get(url, headers=hdrs)
    sha = r.json().get('sha') if r.status_code==200 else None
    payload = {'message': f'Update {filename} — {datetime.now().strftime("%d %b %Y %H:%M")}',
               'content': base64.b64encode(content.encode()).decode(), 'branch': branch}
    if sha: payload['sha'] = sha
    r = requests.put(url, headers=hdrs, json=payload)
    status = 'OK' if r.status_code in (200,201) else f'ERROR {r.status_code}: {r.text[:150]}'
    print(f'  {filename}: {status}')
    return r.status_code in (200,201)

if GITHUB_TOKEN == 'PASTE_YOUR_TOKEN_HERE':
    print('No token — skipping push.')
else:
    print('Pushing to GitHub...')
    push_to_github('index.html', html, GITHUB_TOKEN, GITHUB_REPO)
    print('Done! Live at: https://ramezelias1.github.io/elias-intelligence-research/')